# EEGSynthesizer — generación reproducible y contrato morfológico

Este notebook implementa el generador paramétrico completo. Las etiquetas
`generalized_absence` y `focal_temporal` identifican escenarios sintéticos internos;
no son diagnósticos ni modelos biofísicos completos de una crisis clínica.

## Ejecución reproducible

1. Ejecute primero `1_1_EEGSynthesizer_VALIDATION.ipynb` con `NOTEBOOK_MODE="prepare_dev"`.
2. Mantenga `GENERATION_MODE="full"` y `CALIBRATION_MODE="frozen"` en la primera celda.
3. Use **Run All**. Si el dataset publicado ya existe y sus hashes son válidos, se
   reutiliza; si no existe, el baseline seleccionado se reconstruye desde parámetros,
   semillas y perfil de desarrollo públicos.

`CALIBRATION_MODE="frozen"` reproduce la decisión publicada y auditable. El modo
`recompute` es un experimento nuevo que vuelve a comparar baseline y candidato usando
exclusivamente desarrollo; nunca utiliza las cohortes externas.

Principios: voltios internamente y microvoltios en persistencia; PTP operacional final
150–600 µV; lognormal truncada sin acumulación por `clip`; punta–onda frontocentral;
ritmo focal lateralizado con evolución; degradación de la señal completa; semillas por
paciente; tres pilotos; promoción atómica y separación por paciente.

Base morfológica: Seneviratne et al., *Clinical Neurophysiology* 2016,
DOI 10.1016/j.clinph.2015.08.015; Lee et al., *Epilepsia Open* 2025,
DOI 10.1002/epi4.70052. La validación es operacional y de investigación, no clínica.


In [1]:
# BLOQUE 0 — imports, rutas y configuración congelada
import gc
import hashlib
import json
import importlib.metadata
import math
import os
import platform
import shutil
import sys
import time
from pathlib import Path

import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import scipy
from numpy.lib.format import open_memmap
from scipy import optimize, signal, stats
from sklearn.model_selection import train_test_split

# Configuración visible para Jupyter. Las variables del sistema son opcionales.
GENERATION_MODE = "full"       # pilot | full
CALIBRATION_MODE = "frozen"   # frozen | recompute
GENERATION_MODE = os.environ.get("EEGSYNTH_MODE", GENERATION_MODE).strip().lower()
CALIBRATION_MODE = os.environ.get("EEGSYNTH_CALIBRATION_MODE", CALIBRATION_MODE).strip().lower()
if GENERATION_MODE not in {"pilot","full"}: raise ValueError("GENERATION_MODE debe ser pilot o full")
if CALIBRATION_MODE not in {"frozen","recompute"}: raise ValueError("CALIBRATION_MODE debe ser frozen o recompute")

PROJECT_ROOT = Path.cwd().resolve()
expected={"1_0_EEGSynthesizer_DATASET.ipynb","1_1_EEGSynthesizer_VALIDATION.ipynb","requirements.txt"}
missing=sorted(name for name in expected if not (PROJECT_ROOT/name).exists())
if missing: raise RuntimeError("Abra Jupyter desde la raíz del repositorio; faltan: "+", ".join(missing))
FINAL_DIR = PROJECT_ROOT / "dataset_eeg_final"
WORK_ROOT = PROJECT_ROOT / "tmp" / "eegsynth_rebuild"
CANDIDATE_DIR = WORK_ROOT / "dataset_eeg_candidate"
WORK_ROOT.mkdir(parents=True, exist_ok=True)

FS = 250
DURATION = 120
N_SAMPLES = FS * DURATION
WIN_SEC = 2.0
STEP_SEC = 2.0
WIN_PTS = int(FS * WIN_SEC)
STEP_PTS = int(FS * STEP_SEC)
WINDOWS_PER_PATIENT = 1 + (N_SAMPLES - WIN_PTS) // STEP_PTS
N_PATIENTS = 3000
PILOT_SEEDS = (42, 142, 242)
PILOT_PATIENTS = 100
FINAL_SEED = 42
VAL_FRAC = 0.15
TEST_FRAC = 0.15
BIN_THR = 0.50

CH_NAMES = ["Fp1","Fp2","F7","F3","Fz","F4","F8",
            "T3","C3","Cz","C4","T4","T5","P3","Pz","P4","T6","O1","O2"]
N_CHANNELS = len(CH_NAMES)

# Contrato operacional final observable. No se presenta como rango clínico universal.
SEIZ_TARGET_MEDIAN_UV = 300.0
SEIZ_TARGET_SIGMA_LOG = 0.28
SEIZ_TARGET_MIN_UV = 150.0
SEIZ_TARGET_MAX_UV = 600.0
TARGET_REL_TOL = 0.05

BASE_AMP_MEDIAN_V = 45e-6
BASE_AMP_SIGMA_LOG = 0.40
BASE_AMP_MIN_V = 20e-6
BASE_AMP_MAX_V = 120e-6
BETA_MU, BETA_SIGMA, BETA_MIN, BETA_MAX = 2.0, 0.40, 1.2, 3.0
WHITE_NOISE_FRAC = 0.06
COMMON_FIELD_MIX = (0.18, 0.42)
TOPO_JITTER = (0.85, 1.15)
FOCAL_F0_RANGE = (6.0, 12.0)
FOCAL_F1_RANGE = (3.0, 6.0)
FOCAL_CONTRALATERAL_WEIGHT = 0.05
ALPHA_FREQ = (8.0, 12.0)
ALPHA_AMP_V = (20e-6, 50e-6)
ALPHA_PROB = 0.75
BLINK_PROB = 0.30
BLINK_AMP_V = (40e-6, 120e-6)
EMG_PROBS = (0.60, 0.30, 0.10)
DROPOUT_PROB = 0.05
CO_OCCUR_PROB = 0.15

DEV_PROFILE_DIR = PROJECT_ROOT / "dataset_doctorado_final" / "validation_q1_assets"
USE_DEV_PROFILE = os.environ.get("EEGSYNTH_USE_DEV_PROFILE", "1") == "1"
RUN_FULL = GENERATION_MODE == "full"
STATE_PATH = PROJECT_ROOT / "dataset_doctorado_final" / "reproducibility_state.json"
CONFIG = {
    "schema_version": "4.0",
    "fs_hz": FS, "duration_s": DURATION, "window_s": WIN_SEC, "step_s": STEP_SEC,
    "n_patients": N_PATIENTS, "final_seed": FINAL_SEED,
    "class_counts": {"none": 1500, "generalized_absence": 750, "focal_temporal": 750},
    "target_ptp_uv": {"distribution": "truncated_lognormal", "median": SEIZ_TARGET_MEDIAN_UV,
                      "sigma_log": SEIZ_TARGET_SIGMA_LOG, "min": SEIZ_TARGET_MIN_UV,
                      "max": SEIZ_TARGET_MAX_UV, "relative_tolerance": TARGET_REL_TOL},
    "scenario_semantics": "internal_parametric_not_clinical_diagnoses",
    "degradation_order": "after_complete_composition",
    "pilot_seeds": list(PILOT_SEEDS), "pilot_patients_per_seed": PILOT_PATIENTS,
}
CONFIG["generation_mode"]=GENERATION_MODE
CONFIG["calibration_mode"]=CALIBRATION_MODE
versions={name:importlib.metadata.version(name) for name in ("numpy","pandas","scipy","scikit-learn")}
print(json.dumps(CONFIG, indent=2, ensure_ascii=False))
print("Entorno:",sys.version.split()[0],versions)
print(f"Espacio libre: {shutil.disk_usage(PROJECT_ROOT).free/(1024**3):.2f} GiB")


{
  "schema_version": "4.0",
  "fs_hz": 250,
  "duration_s": 120,
  "window_s": 2.0,
  "step_s": 2.0,
  "n_patients": 3000,
  "final_seed": 42,
  "class_counts": {
    "none": 1500,
    "generalized_absence": 750,
    "focal_temporal": 750
  },
  "target_ptp_uv": {
    "distribution": "truncated_lognormal",
    "median": 300.0,
    "sigma_log": 0.28,
    "min": 150.0,
    "max": 600.0,
    "relative_tolerance": 0.05
  },
  "scenario_semantics": "internal_parametric_not_clinical_diagnoses",
  "degradation_order": "after_complete_composition",
  "pilot_seeds": [
    42,
    142,
    242
  ],
  "pilot_patients_per_seed": 100,
  "generation_mode": "full",
  "calibration_mode": "frozen"
}
Entorno: 3.13.7 {'numpy': '2.4.2', 'pandas': '3.0.1', 'scipy': '1.17.1', 'scikit-learn': '1.8.0'}
Espacio libre: 358.03 GiB


In [2]:
# BLOQUE 1 — utilidades numéricas y morfologías
def stable_seed(root_seed, patient_id, attempt=0):
    return int(np.random.SeedSequence([int(root_seed), int(patient_id), int(attempt)]).generate_state(1)[0])


def truncated_lognormal_uv(rng):
    mu = math.log(SEIZ_TARGET_MEDIAN_UV)
    lo = (math.log(SEIZ_TARGET_MIN_UV) - mu) / SEIZ_TARGET_SIGMA_LOG
    hi = (math.log(SEIZ_TARGET_MAX_UV) - mu) / SEIZ_TARGET_SIGMA_LOG
    u = rng.uniform(stats.norm.cdf(lo), stats.norm.cdf(hi))
    return float(math.exp(mu + SEIZ_TARGET_SIGMA_LOG * stats.norm.ppf(u)))


def clipped_lognormal(rng, median, sigma, low, high):
    return float(np.clip(rng.lognormal(math.log(median), sigma), low, high))


def colored_noise(rng, n_channels, n_samples, beta):
    u = rng.normal(size=(n_channels, n_samples))
    spec = np.fft.rfft(u, axis=1)
    freqs = np.fft.rfftfreq(n_samples)
    scale = np.ones_like(freqs)
    scale[1:] = freqs[1:] ** (-beta / 2.0)
    x = np.fft.irfft(spec * scale[None, :], n=n_samples, axis=1)
    x -= x.mean(axis=1, keepdims=True)
    x /= x.std(axis=1, keepdims=True) + 1e-12
    return x


def smooth_gate(n, fs, ramp_s=0.75):
    gate = np.ones(n, dtype=np.float64)
    r = min(int(round(ramp_s * fs)), n // 2)
    if r > 1:
        edge = np.sin(np.linspace(0, np.pi / 2, r)) ** 2
        gate[:r] = edge
        gate[-r:] = edge[::-1]
    return gate


def cyclic_distance(phase01, center):
    return ((phase01 - center + 0.5) % 1.0) - 0.5


def generalized_spike_wave(rng, n, fs):
    freq = float(rng.uniform(2.5, 4.0))
    phase0 = float(rng.uniform(0, 2 * np.pi))
    phase = phase0 + 2 * np.pi * freq * np.arange(n) / fs
    p = (phase / (2 * np.pi)) % 1.0
    spike = np.exp(-0.5 * (cyclic_distance(p, 0.08) / 0.030) ** 2)
    slow = -0.72 * np.exp(-0.5 * (cyclic_distance(p, 0.37) / 0.145) ** 2)
    wave = (spike + slow) * smooth_gate(n, fs, 0.6)
    sos = signal.butter(4, 6.0, btype="lowpass", fs=fs, output="sos")
    wave = signal.sosfiltfilt(sos, wave)
    wave -= wave.mean()
    wave /= np.ptp(wave) + 1e-12
    return wave, {"freq_start_hz": freq, "freq_end_hz": freq, "freq_slope_hz_s": 0.0}


def focal_evolving_rhythm(rng, n, fs):
    # Patrón operacional inspirado en la evolución descrita para crisis focales:
    # inicio más rápido y de menor amplitud, seguido de enlentecimiento y aumento.
    f0 = float(rng.uniform(*FOCAL_F0_RANGE))
    upper_f1 = min(FOCAL_F1_RANGE[1], f0 - 0.75)
    f1 = float(rng.uniform(FOCAL_F1_RANGE[0], max(FOCAL_F1_RANGE[0] + 1e-6, upper_f1)))
    t = np.arange(n) / fs
    progress = np.linspace(0.0, 1.0, n)
    freq = f0 + (f1 - f0) * (3 * progress**2 - 2 * progress**3)
    phase = float(rng.uniform(0, 2*np.pi)) + 2*np.pi*np.cumsum(freq) / fs
    carrier = np.sin(phase) + 0.24*np.sin(2*phase + rng.uniform(-0.5, 0.5))
    envelope = (0.32 + 0.68 * (3*progress**2 - 2*progress**3)) * smooth_gate(n, fs, 0.8)
    wave = carrier * envelope
    # Transitorios agudos escasos, siempre ligados a la fase, no tren idéntico punta–onda.
    for k in range(1, max(2, int(n / fs / 4))):
        center = int((k * n / max(2, int(n / fs / 4))) + rng.integers(-fs//5, fs//5))
        if 4 <= center < n-5 and rng.random() < 0.55:
            width = int(rng.integers(max(3, fs//80), max(4, fs//35)))
            a, b = max(0, center-3*width), min(n, center+3*width+1)
            q = np.arange(a, b) - center
            wave[a:b] += rng.uniform(0.10, 0.24) * np.exp(-0.5*(q/width)**2)
    sos = signal.butter(4, [1.0, 20.0], btype="bandpass", fs=fs, output="sos")
    wave = signal.sosfiltfilt(sos, wave)
    wave -= wave.mean()
    wave /= np.ptp(wave) + 1e-12
    return wave, {"freq_start_hz": f0, "freq_end_hz": f1,
                  "freq_slope_hz_s": (f1-f0) / max(n/fs, 1e-9)}


GENERALIZED_WEIGHTS = {
    "Fp1":1.10,"Fp2":1.10,"F7":0.60,"F3":1.00,"Fz":1.10,"F4":1.00,"F8":0.60,
    "T3":0.30,"C3":0.40,"Cz":0.50,"C4":0.40,"T4":0.30,"T5":0.10,"P3":0.10,
    "Pz":0.20,"P4":0.10,"T6":0.10,"O1":0.05,"O2":0.05,
}
FOCAL_LEFT_WEIGHTS = {"T3":1.20,"F7":0.80,"T5":0.80,"C3":0.50,"Fp1":0.40,"F3":0.40,"P3":0.30,"O1":0.10}
FOCAL_RIGHT_WEIGHTS = {"T4":1.20,"F8":0.80,"T6":0.80,"C4":0.50,"Fp2":0.40,"F4":0.40,"P4":0.30,"O2":0.10}
PRIMARY_GENERALIZED = ["Fp1","Fp2","F3","Fz","F4"]
PRIMARY_FOCAL_LEFT = ["T3","F7","T5","C3"]
PRIMARY_FOCAL_RIGHT = ["T4","F8","T6","C4"]


MEASURE_SOS = signal.butter(4, [1.0, 30.0], btype="bandpass", fs=FS, output="sos")


def filtered_ptp_uv(segment_v, channel_indices):
    if segment_v.shape[1] < 3 * 12:
        return float("nan")
    xf = signal.sosfiltfilt(MEASURE_SOS, segment_v[channel_indices], axis=1)
    return float(np.median(np.ptp(xf, axis=1)) * 1e6)


def dominant_frequency(x, fs, fmin=1.0, fmax=20.0):
    f, p = signal.welch(np.asarray(x, dtype=np.float64), fs=fs,
                        nperseg=min(len(x), 2*fs))
    mask = (f >= fmin) & (f <= fmax)
    return float(f[mask][np.argmax(p[mask])]) if np.any(mask) else float("nan")


In [3]:
# BLOQUE 2 — sintetizador completo y calibración del PTP final
class EEGSynthesizer:
    def __init__(self, fs=FS, duration=DURATION):
        self.fs = fs
        self.duration = duration
        self.n_samples = int(fs * duration)

    def _background(self, rng):
        beta = float(np.clip(rng.normal(BETA_MU, BETA_SIGMA), BETA_MIN, BETA_MAX))
        amp = clipped_lognormal(rng, BASE_AMP_MEDIAN_V, BASE_AMP_SIGMA_LOG,
                                BASE_AMP_MIN_V, BASE_AMP_MAX_V)
        independent = colored_noise(rng, N_CHANNELS, self.n_samples, beta)
        common = colored_noise(rng, 1, self.n_samples, beta)[0]
        mix = float(rng.uniform(*COMMON_FIELD_MIX))
        x = (math.sqrt(1-mix) * independent + math.sqrt(mix) * common[None, :]) * amp
        x += rng.normal(0, amp * WHITE_NOISE_FRAC, size=x.shape)
        # Ritmo alfa posterior con variación leve por paciente.
        alpha_f = float(rng.uniform(*ALPHA_FREQ))
        alpha_a = float(rng.uniform(*ALPHA_AMP_V))
        phase = float(rng.uniform(0, 2*np.pi))
        alpha = np.sin(2*np.pi*alpha_f*np.arange(self.n_samples)/self.fs + phase) * alpha_a
        alpha_weights = {"O1":1.0,"O2":1.0,"P3":0.55,"Pz":0.60,"P4":0.55,"T5":0.25,"T6":0.25}
        for ci, ch in enumerate(CH_NAMES):
            if ch in alpha_weights and rng.random() < ALPHA_PROB:
                x[ci] += alpha * alpha_weights[ch] * rng.uniform(0.85,1.15)
        return x, beta, amp, alpha_f

    def _add_blink(self, rng, x, artifact_mask, center=None):
        length = int(rng.uniform(0.35, 0.9) * self.fs)
        if center is None:
            start = int(rng.integers(0, max(1, self.n_samples-length)))
        else:
            start = int(np.clip(center - length//2, 0, self.n_samples-length))
        q = np.linspace(-2.8, 2.8, length)
        pulse = np.exp(-0.5*q*q) * rng.uniform(*BLINK_AMP_V)
        for ch, w in {"Fp1":1.0,"Fp2":1.0,"F7":0.30,"F3":0.35,"F4":0.35,"F8":0.30}.items():
            x[CH_NAMES.index(ch), start:start+length] += pulse * w * rng.uniform(0.85,1.15)
        artifact_mask[start:start+length] = True

    def _add_emg(self, rng, x, artifact_mask, start=None, end=None, strength=1.0):
        if start is None:
            length = int(rng.uniform(0.4, 2.0) * self.fs)
            start = int(rng.integers(0, max(1, self.n_samples-length)))
            end = start + length
        length = int(end-start)
        if length < 10:
            return
        sos = signal.butter(4, [25, 90], btype="bandpass", fs=self.fs, output="sos")
        channels = rng.choice(N_CHANNELS, size=int(rng.integers(2, 7)), replace=False)
        for ci in channels:
            noise = signal.sosfiltfilt(sos, rng.normal(size=length))
            noise /= np.std(noise) + 1e-12
            x[ci, start:end] += noise * rng.uniform(8e-6, 28e-6) * strength
        artifact_mask[start:end] = True

    def _confuser(self, rng, x):
        if rng.random() >= 0.50:
            return False
        dur = float(rng.uniform(1.0, 3.0)); n = int(dur*self.fs)
        start = int(rng.integers(0, self.n_samples-n))
        f = float(rng.uniform(4.0, 8.0)); amp = float(rng.uniform(40e-6, 120e-6))
        wave = np.sin(2*np.pi*f*np.arange(n)/self.fs + rng.uniform(0,2*np.pi))*smooth_gate(n,self.fs,0.25)*amp
        channels = rng.choice(N_CHANNELS, size=int(rng.integers(2,6)), replace=False)
        for ci in channels: x[ci,start:start+n] += wave*rng.uniform(0.4,1.0)
        return True

    def _event_component(self, rng, scenario, start, duration):
        i0 = int(round(start*self.fs)); n = int(round(duration*self.fs)); i1 = min(self.n_samples, i0+n)
        n = i1-i0
        unit = np.zeros((N_CHANNELS, self.n_samples), dtype=np.float64)
        if scenario == "generalized_absence":
            wave, evo = generalized_spike_wave(rng, n, self.fs)
            weights = GENERALIZED_WEIGHTS
            lateral = "bilateral"
            primary = PRIMARY_GENERALIZED
            default_weight = 0.05
        elif scenario == "focal_temporal":
            wave, evo = focal_evolving_rhythm(rng, n, self.fs)
            lateral = "left" if rng.random() < 0.5 else "right"
            weights = FOCAL_LEFT_WEIGHTS if lateral == "left" else FOCAL_RIGHT_WEIGHTS
            default_weight = FOCAL_CONTRALATERAL_WEIGHT
            primary = PRIMARY_FOCAL_LEFT if lateral == "left" else PRIMARY_FOCAL_RIGHT
        else:
            raise ValueError(scenario)
        actual_weights = []
        for ci, ch in enumerate(CH_NAMES):
            w = float(weights.get(ch, default_weight) * rng.uniform(*TOPO_JITTER))
            unit[ci, i0:i1] = wave * w
            actual_weights.append(w)
        return unit, i0, i1, lateral, primary, actual_weights, evo

    def _measurement_slice(self, i0, i1, artifact_mask):
        margin = int(0.20 * (i1-i0))
        lo, hi = i0+margin, i1-margin
        win = min(WIN_PTS, max(0, hi-lo))
        if win < WIN_PTS:
            return slice(max(i0, (i0+i1-WIN_PTS)//2), min(i1, (i0+i1+WIN_PTS)//2)), False
        candidates = np.arange(lo, hi-WIN_PTS+1, max(1, WIN_PTS//4))
        clean = [int(s) for s in candidates if not artifact_mask[s:s+WIN_PTS].any()]
        if clean:
            s = clean[len(clean)//2]
            return slice(s, s+WIN_PTS), True
        s = int(candidates[len(candidates)//2])
        return slice(s, s+WIN_PTS), False

    def generate_patient(self, scenario, root_seed, patient_id, max_attempts=10):
        for attempt in range(1, max_attempts+1):
            patient_seed = stable_seed(root_seed, patient_id, attempt-1)
            rng = np.random.default_rng(patient_seed)
            base, beta, base_amp, alpha_f = self._background(rng)
            artifact_mask = np.zeros(self.n_samples, dtype=bool)
            blink = rng.random() < BLINK_PROB
            if blink: self._add_blink(rng, base, artifact_mask)
            emg_level = int(rng.choice([0,1,2], p=EMG_PROBS))
            for _ in range(emg_level): self._add_emg(rng, base, artifact_mask)

            labels = np.zeros(self.n_samples, dtype=np.float32)
            unit = np.zeros_like(base)
            target_uv = observed_uv = rel_error = float("nan")
            start = duration = 0.0; lateral = "none"; primary = []
            weights = [0.0]*N_CHANNELS
            evo = {"freq_start_hz":float("nan"),"freq_end_hz":float("nan"),"freq_slope_hz_s":float("nan")}
            confuser = False; co_artifact = False; measure_clean = True

            if scenario == "none":
                confuser = self._confuser(rng, base)
                i0 = i1 = 0
            else:
                if scenario == "generalized_absence":
                    duration = float(np.clip(rng.lognormal(math.log(8.0),0.45), 3.0, 30.0))
                else:
                    duration = float(np.clip(rng.lognormal(math.log(25.0),0.50), 8.0, 75.0))
                start = float(rng.uniform(5.0, max(5.01, DURATION-duration-5.0)))
                unit, i0, i1, lateral, primary, weights, evo = self._event_component(rng,scenario,start,duration)
                labels[i0:i1] = 1.0
                if rng.random() < CO_OCCUR_PROB:
                    co_artifact = True
                    center = int(rng.integers(i0, max(i0+1,i1)))
                    if rng.random() < 0.5: self._add_blink(rng,base,artifact_mask,center=center)
                    else:
                        n = int(rng.uniform(0.4,1.5)*self.fs)
                        self._add_emg(rng,base,artifact_mask,max(i0,center-n//2),min(i1,center+n//2),1.2)

            degradation = np.ones(N_CHANNELS, dtype=np.float64)
            degraded = []
            for ci,ch in enumerate(CH_NAMES):
                if rng.random() < DROPOUT_PROB:
                    degradation[ci] = rng.uniform(0.05,0.15)
                    degraded.append(ch)
            base_d = base * degradation[:,None]
            unit_d = unit * degradation[:,None]

            amplitude_ok = True
            morphology_ok = True
            gen_freq_ok = focal_evolution_ok = laterality_ok = symmetry_ok = True
            scale_v = 0.0
            if scenario != "none":
                target_uv = truncated_lognormal_uv(rng)
                mslice, measure_clean = self._measurement_slice(i0,i1,artifact_mask)
                primary_idx = [CH_NAMES.index(c) for c in primary if c not in degraded]
                if not primary_idx:
                    primary_idx = [CH_NAMES.index(c) for c in primary]
                bf = signal.sosfiltfilt(MEASURE_SOS, base_d[primary_idx,mslice], axis=1)
                uf = signal.sosfiltfilt(MEASURE_SOS, unit_d[primary_idx,mslice], axis=1)
                unit_ptp = float(np.median(np.ptp(uf,axis=1))*1e6)
                guess = target_uv / max(unit_ptp,1e-9)
                upper = max(guess*3.0, 5e-4)
                def objective(a):
                    val = float(np.median(np.ptp(bf+a*uf,axis=1))*1e6)
                    return abs(val-target_uv)
                opt = optimize.minimize_scalar(objective,bounds=(0.0,upper),method="bounded",
                                               options={"xatol":1e-8,"maxiter":60})
                scale_v = float(opt.x)
                observed_uv = filtered_ptp_uv(base_d[:,mslice]+scale_v*unit_d[:,mslice], primary_idx)
                rel_error = abs(observed_uv-target_uv)/target_uv
                amplitude_ok = bool(np.isfinite(observed_uv) and SEIZ_TARGET_MIN_UV <= observed_uv <= SEIZ_TARGET_MAX_UV and rel_error <= TARGET_REL_TOL)

                seg = base_d[:,i0:i1] + scale_v*unit_d[:,i0:i1]
                if scenario == "generalized_absence":
                    fz = signal.sosfiltfilt(MEASURE_SOS, seg[CH_NAMES.index("Fz")])
                    freq_obs = dominant_frequency(fz,self.fs,2.0,8.0)
                    gen_freq_ok = 2.5 <= freq_obs <= 4.0
                    pairs=[]
                    for l,r in [("Fp1","Fp2"),("F3","F4")]:
                        if l not in degraded and r not in degraded:
                            pl=np.ptp(signal.sosfiltfilt(MEASURE_SOS,seg[CH_NAMES.index(l)]))
                            pr=np.ptp(signal.sosfiltfilt(MEASURE_SOS,seg[CH_NAMES.index(r)]))
                            pairs.append(abs(pl-pr)/(max(pl,pr)+1e-12))
                    symmetry_ok = bool(not pairs or np.median(pairs) <= 0.35)
                else:
                    nseg=seg.shape[1]; third=max(WIN_PTS,nseg//3)
                    dom_ch = primary[0]; x=signal.sosfiltfilt(MEASURE_SOS,seg[CH_NAMES.index(dom_ch)])
                    f_first=dominant_frequency(x[:third],self.fs,1.0,20.0)
                    f_last=dominant_frequency(x[-third:],self.fs,1.0,20.0)
                    rms_first=float(np.sqrt(np.mean(x[:third]**2))); rms_last=float(np.sqrt(np.mean(x[-third:]**2)))
                    focal_evolution_ok = bool(f_first > f_last and rms_last > rms_first)
                    left=np.median([np.ptp(seg[CH_NAMES.index(c)]) for c in PRIMARY_FOCAL_LEFT])
                    right=np.median([np.ptp(seg[CH_NAMES.index(c)]) for c in PRIMARY_FOCAL_RIGHT])
                    laterality_ok = bool((lateral=="left" and left>1.15*right) or (lateral=="right" and right>1.15*left))
                morphology_ok = bool(gen_freq_ok and focal_evolution_ok and laterality_ok and symmetry_ok)

            data_v = base_d + scale_v * unit_d
            if scenario == "none" or (amplitude_ok and morphology_ok):
                meta = {
                    "patient_id":int(patient_id),"patient_seed":patient_seed,"attempts":attempt,
                    "scenario":scenario,"event_id":f"p{patient_id:04d}_e0" if scenario!="none" else "",
                    "start_s":start,"duration_s":duration,"lateralization":lateral,
                    **evo,"target_ptp_uv":target_uv,"observed_ptp_uv":observed_uv,"target_rel_error":rel_error,
                    "amplitude_ok":bool(amplitude_ok),"morphology_ok":bool(morphology_ok),
                    "generalized_frequency_ok":bool(gen_freq_ok),"symmetry_ok":bool(symmetry_ok),
                    "focal_evolution_ok":bool(focal_evolution_ok),"laterality_ok":bool(laterality_ok),
                    "measurement_clean":bool(measure_clean),"primary_channels":"|".join(primary),
                    "topographic_weights":"|".join(f"{x:.6g}" for x in weights),
                    "blink":bool(blink),"emg_bursts":emg_level,"co_artifact":bool(co_artifact),
                    "artifact_fraction":float(artifact_mask.mean()),"confuser":bool(confuser),
                    "degraded_channels":"|".join(degraded),
                    "degradation_factors":"|".join(f"{x:.6g}" for x in degradation),
                    "beta":beta,"base_amp_v":base_amp,"alpha_freq_hz":alpha_f,
                    "accepted":True,
                }
                return (data_v*1e6).T.astype(np.float32), labels, artifact_mask, meta
        raise RuntimeError(f"Paciente {patient_id} ({scenario}) no cumplió el contrato en {max_attempts} intentos")


In [4]:
# BLOQUE 3 — propuesta determinista basada exclusivamente en desarrollo
def cal_sha256(path):
    return hashlib.sha256(Path(path).read_bytes()).hexdigest()

CAL_FEATURE_NAMES = ["ptp_med","ptp_p95","std_med","std_p95","bp1_4","bp4_8","bp8_13","bp13_30",
                     "ratio_2_6__6_20","beta","Hspec","corr_abs_mean","corr_abs_p95","dom_freq",
                     "freq_first","freq_last","rms_last_first","spatial_concentration","laterality_abs"]
CAL_DOMAINS = {"background":[0,1,4,5,6,7,9,10],"temporal":[8,13,14,15,16],"spatial":[11,12,17,18]}
CAL_BIPOLAR = [("Fp1","F7"),("F7","T3"),("T3","T5"),("T5","O1"),("Fp1","F3"),("F3","C3"),("C3","P3"),("P3","O1"),
               ("Fp2","F4"),("F4","C4"),("C4","P4"),("P4","O2"),("Fp2","F8"),("F8","T4"),("T4","T6"),("T6","O2"),("Fz","Cz"),("Cz","Pz")]

def cal_bipolar(w):
    idx={c:i for i,c in enumerate(CH_NAMES)}
    return np.stack([w[:,idx[a]]-w[:,idx[b]] for a,b in CAL_BIPOLAR],axis=1)

def cal_feature(w):
    x=np.asarray(w,dtype=np.float64); ptp=np.ptp(x,axis=0); sd=x.std(0)
    f,pch=signal.welch(x,fs=FS,nperseg=min(256,len(x)),axis=0); p=pch.mean(1)
    def bp(a,b):
        m=(f>=a)&(f<b); return float(np.trapezoid(p[m],f[m])) if m.sum()>=2 else 0.0
    m=(f>=1)&(f<=30)&(p>0); beta=float(np.polyfit(np.log(f[m]),np.log(p[m]),1)[0]) if m.sum()>=3 else 0.0
    pn=p/(p.sum()+1e-12); H=float(-(pn[pn>0]*np.log(pn[pn>0])).sum()/np.log(max(2,len(pn))))
    C=np.corrcoef(x,rowvar=False); cv=np.abs(C[np.triu_indices_from(C,k=1)]); cv=cv[np.isfinite(cv)]
    band=(f>=1)&(f<=20); dom=float(f[band][np.argmax(p[band])]); third=max(FS//2,len(x)//3)
    def dp(q):
        fq,pq=signal.welch(q,fs=FS,nperseg=min(256,len(q)),axis=0); pq=pq.mean(1); mm=(fq>=1)&(fq<=20)
        return float(fq[mm][np.argmax(pq[mm])])
    left=float(ptp[:8].sum()); right=float(ptp[8:16].sum())
    return np.asarray([np.median(ptp),np.percentile(ptp,95),np.median(sd),np.percentile(sd,95),bp(1,4),bp(4,8),bp(8,13),bp(13,30),
                       bp(2,6)/(bp(6,20)+1e-12),beta,H,cv.mean(),np.percentile(cv,95),dom,dp(x[:third]),dp(x[-third:]),
                       np.sqrt(np.mean(x[-third:]**2))/(np.sqrt(np.mean(x[:third]**2))+1e-12),ptp.max()/(ptp.sum()+1e-12),abs(left-right)/(left+right+1e-12)],float)

def scenario_vector(n):
    if n % 4: raise ValueError("El tamaño de cohorte debe ser múltiplo de 4")
    return np.array(["none"]*(n//2)+["generalized_absence"]*(n//4)+["focal_temporal"]*(n//4),dtype=object)

def build_cohort(n,seed):
    rng=np.random.default_rng(seed); scenarios=scenario_vector(n); rng.shuffle(scenarios); ids=np.arange(n)
    train_ids,temp=train_test_split(ids,test_size=VAL_FRAC+TEST_FRAC,random_state=seed,stratify=scenarios)
    val_ids,test_ids=train_test_split(temp,test_size=0.5,random_state=seed,stratify=scenarios[temp])
    split=np.empty(n,dtype=object); split[train_ids]="train"; split[val_ids]="val"; split[test_ids]="test"
    return pd.DataFrame({"patient_id":ids,"scenario":scenarios,"split":split})

BASELINE_PARAMS={"BASE_AMP_MEDIAN_V":BASE_AMP_MEDIAN_V,"BETA_MU":BETA_MU,"BETA_SIGMA":BETA_SIGMA,
                 "COMMON_FIELD_MIX":COMMON_FIELD_MIX,"ALPHA_AMP_V":ALPHA_AMP_V,"ALPHA_PROB":ALPHA_PROB,
                 "TOPO_JITTER":TOPO_JITTER,"FOCAL_F0_RANGE":FOCAL_F0_RANGE,"FOCAL_F1_RANGE":FOCAL_F1_RANGE,"FOCAL_CONTRALATERAL_WEIGHT":FOCAL_CONTRALATERAL_WEIGHT}
DEV_PROFILE_PATH=DEV_PROFILE_DIR/"development_profile.json"; DEV_FEATURE_PATH=DEV_PROFILE_DIR/"dev_feature_profile.csv"; DEV_EVENT_PATH=DEV_PROFILE_DIR/"dev_event_evolution.csv"
FROZEN_AUDIT_PATH=DEV_PROFILE_DIR/"generator_calibration_audit.json"
required_profile=(DEV_PROFILE_PATH,DEV_FEATURE_PATH,DEV_EVENT_PATH)
if not USE_DEV_PROFILE or not all(p.exists() for p in required_profile):
    raise RuntimeError("Falta el perfil de desarrollo 5.1; ejecute primero 1_1 con NOTEBOOK_MODE='prepare_dev'")
DEV_PROFILE=json.loads(DEV_PROFILE_PATH.read_text(encoding="utf-8"))
if DEV_PROFILE.get("schema_version")!="5.1-dev" or set(DEV_PROFILE.get("subjects",[]))!={"chb01","chb02","chb03","chb05","chb06"}:
    raise RuntimeError("Perfil de desarrollo incompatible o con sujetos incorrectos")
DEV_FEATURE=pd.read_csv(DEV_FEATURE_PATH); DEV_EVENT=pd.read_csv(DEV_EVENT_PATH)

def calibration_sample(params):
    global BASE_AMP_MEDIAN_V,BETA_MU,BETA_SIGMA,COMMON_FIELD_MIX,ALPHA_AMP_V,ALPHA_PROB,TOPO_JITTER,FOCAL_F0_RANGE,FOCAL_F1_RANGE,FOCAL_CONTRALATERAL_WEIGHT
    for name,value in params.items(): globals()[name]=tuple(value) if name in {"COMMON_FIELD_MIX","ALPHA_AMP_V","TOPO_JITTER","FOCAL_F0_RANGE","FOCAL_F1_RANGE"} else value
    feats=[]; labels=[]; synth=EEGSynthesizer()
    for seed in PILOT_SEEDS:
        cohort=build_cohort(PILOT_PATIENTS,seed)
        for row in cohort.itertuples(index=False):
            data,lseq,artifact,meta=synth.generate_patient(row.scenario,seed,row.patient_id)
            starts=np.arange(0,N_SAMPLES-WIN_PTS+1,WIN_PTS); fractions=np.asarray([lseq[s:s+WIN_PTS].mean() for s in starts])
            target=0 if row.scenario=="none" else 1; valid=np.flatnonzero(fractions<0.01) if target==0 else np.flatnonzero(fractions>=0.5)
            if not len(valid): continue
            positions=np.unique(np.linspace(0,len(valid)-1,min(3,len(valid))).round().astype(int))
            for pos in positions:
                pick=int(valid[pos]); s=int(starts[pick]); feats.append(cal_feature(cal_bipolar(data[s:s+WIN_PTS]))); labels.append(target)
    return np.asarray(feats),np.asarray(labels,dtype=int)

if CALIBRATION_MODE=="frozen":
    if not FROZEN_AUDIT_PATH.exists(): raise RuntimeError("Falta el snapshot público generator_calibration_audit.json")
    FROZEN_CALIBRATION_AUDIT=json.loads(FROZEN_AUDIT_PATH.read_text(encoding="utf-8"))
    if FROZEN_CALIBRATION_AUDIT.get("development_profile_sha256")!=cal_sha256(DEV_PROFILE_PATH):
        raise RuntimeError("El snapshot de calibración no corresponde al perfil de desarrollo")
    CALIBRATED_PARAMS=FROZEN_CALIBRATION_AUDIT["candidate_parameters"]
    print("Calibración publicada: snapshot verificado; selección =", "candidate" if FROZEN_CALIBRATION_AUDIT["selected_candidate"] else "baseline")
else:
    BASELINE_F,BASELINE_Y=calibration_sample(BASELINE_PARAMS)
    def real_subject_median(cls,name): return float(DEV_FEATURE[(DEV_FEATURE["class"]==cls)&(DEV_FEATURE.feature==name)]["median"].median())
    def syn_median(cls,name): return float(np.median(BASELINE_F[BASELINE_Y==cls,CAL_FEATURE_NAMES.index(name)]))
    ptp_scale=float(np.clip(real_subject_median(0,"ptp_med")/(syn_median(0,"ptp_med")+1e-12),1.0,1.5))
    BASE_AMP_MEDIAN_V=float(np.clip(BASE_AMP_MEDIAN_V*ptp_scale,BASE_AMP_MIN_V,90e-6))
    beta_delta=float(np.clip(syn_median(0,"beta")-real_subject_median(0,"beta"),-0.4,0.4)); BETA_MU=float(np.clip(BETA_MU+beta_delta,BETA_MIN,BETA_MAX))
    real_beta_iqr=float(DEV_FEATURE[(DEV_FEATURE["class"]==0)&(DEV_FEATURE.feature=="beta")]["iqr"].median()); BETA_SIGMA=float(np.clip(max(BETA_SIGMA,real_beta_iqr/1.349),0.25,0.65))
    alpha_scale=float(np.clip(np.sqrt(real_subject_median(0,"bp8_13")/(syn_median(0,"bp8_13")+1e-12)),0.5,1.0)); ALPHA_AMP_V=tuple(float(v*alpha_scale) for v in ALPHA_AMP_V); ALPHA_PROB=float(np.clip(ALPHA_PROB*alpha_scale,0.35,0.75))
    corr_scale=float(np.clip(real_subject_median(0,"corr_abs_mean")/(syn_median(0,"corr_abs_mean")+1e-12),0.75,1.25)); COMMON_FIELD_MIX=tuple(float(np.clip(v*corr_scale,0.08,0.60)) for v in COMMON_FIELD_MIX)
    real_spread=float(DEV_FEATURE.groupby(["class","feature"])["iqr"].median().median()); syn_spread=float(np.median([stats.iqr(BASELINE_F[BASELINE_Y==c,j]) for c in (0,1) for j in range(len(CAL_FEATURE_NAMES))])); TOPO_JITTER=(0.75,1.25) if real_spread>syn_spread else TOPO_JITTER
    valid_events=DEV_EVENT[DEV_EVENT.n_windows>=3]
    if len(valid_events):
        f0=float(np.clip(valid_events.freq_start_hz.median(),4.0,10.0)); f1=float(np.clip(valid_events.freq_end_hz.median(),2.5,min(6.0,f0-0.75)))
        FOCAL_F0_RANGE=(max(4.0,f0-2.0),min(12.0,f0+2.0)); FOCAL_F1_RANGE=(max(2.5,f1-1.25),min(6.0,f1+1.25))
    BASE_AMP_MEDIAN_V=BASELINE_PARAMS["BASE_AMP_MEDIAN_V"]; BETA_MU=BASELINE_PARAMS["BETA_MU"]; BETA_SIGMA=BASELINE_PARAMS["BETA_SIGMA"]
    ALPHA_AMP_V=BASELINE_PARAMS["ALPHA_AMP_V"]; ALPHA_PROB=BASELINE_PARAMS["ALPHA_PROB"]
    COMMON_FIELD_MIX=tuple(float(np.clip(v*corr_scale,0.08,0.60)) for v in BASELINE_PARAMS["COMMON_FIELD_MIX"]); TOPO_JITTER=BASELINE_PARAMS["TOPO_JITTER"]
    lat_ratio=float(np.clip(syn_median(1,"laterality_abs")/(real_subject_median(1,"laterality_abs")+1e-12),1.0,5.0))
    FOCAL_CONTRALATERAL_WEIGHT=float(np.clip(BASELINE_PARAMS["FOCAL_CONTRALATERAL_WEIGHT"]*lat_ratio,0.05,0.25))
    CALIBRATED_PARAMS={"BASE_AMP_MEDIAN_V":BASE_AMP_MEDIAN_V,"BETA_MU":BETA_MU,"BETA_SIGMA":BETA_SIGMA,
                       "COMMON_FIELD_MIX":COMMON_FIELD_MIX,"ALPHA_AMP_V":ALPHA_AMP_V,"ALPHA_PROB":ALPHA_PROB,
                       "TOPO_JITTER":TOPO_JITTER,"FOCAL_F0_RANGE":FOCAL_F0_RANGE,"FOCAL_F1_RANGE":FOCAL_F1_RANGE,"FOCAL_CONTRALATERAL_WEIGHT":FOCAL_CONTRALATERAL_WEIGHT}

CONFIG["development_profile_sha256"]=cal_sha256(DEV_PROFILE_PATH); CONFIG["calibration_candidate_parameters"]=CALIBRATED_PARAMS
print("BASELINE PARAMS",json.dumps(BASELINE_PARAMS,indent=2)); print("CANDIDATE PARAMS",json.dumps(CALIBRATED_PARAMS,indent=2))


Calibración publicada: snapshot verificado; selección = baseline
BASELINE PARAMS {
  "BASE_AMP_MEDIAN_V": 4.5e-05,
  "BETA_MU": 2.0,
  "BETA_SIGMA": 0.4,
  "COMMON_FIELD_MIX": [
    0.18,
    0.42
  ],
  "ALPHA_AMP_V": [
    2e-05,
    5e-05
  ],
  "ALPHA_PROB": 0.75,
  "TOPO_JITTER": [
    0.85,
    1.15
  ],
  "FOCAL_F0_RANGE": [
    6.0,
    12.0
  ],
  "FOCAL_F1_RANGE": [
    3.0,
    6.0
  ],
  "FOCAL_CONTRALATERAL_WEIGHT": 0.05
}
CANDIDATE PARAMS {
  "BASE_AMP_MEDIAN_V": 4.5e-05,
  "BETA_MU": 2.0,
  "BETA_SIGMA": 0.4,
  "COMMON_FIELD_MIX": [
    0.1659718966704249,
    0.3872677588976581
  ],
  "ALPHA_AMP_V": [
    2e-05,
    5e-05
  ],
  "ALPHA_PROB": 0.75,
  "TOPO_JITTER": [
    0.85,
    1.15
  ],
  "FOCAL_F0_RANGE": [
    4.0,
    6.0
  ],
  "FOCAL_F1_RANGE": [
    2.5,
    3.75
  ],
  "FOCAL_CONTRALATERAL_WEIGHT": 0.13862214758769617
}


In [5]:
# BLOQUE 4 — pilotos independientes y reproducibilidad
def pilot_run(seed, include_digest=True):
    synth=EEGSynthesizer(); cohort=build_cohort(PILOT_PATIENTS,seed)
    metas=[]; digest=hashlib.sha256()
    for row in cohort.itertuples(index=False):
        data,labels,artifact,meta=synth.generate_patient(row.scenario,seed,row.patient_id)
        meta["split"]=row.split; metas.append(meta)
        if include_digest:
            digest.update(data.tobytes()); digest.update(labels.tobytes())
    df=pd.DataFrame(metas)
    ict=df[df.scenario!="none"]
    gen=df[df.scenario=="generalized_absence"]
    foc=df[df.scenario=="focal_temporal"]
    report={
        "seed":seed,"n":len(df),"finite":bool(np.isfinite(ict.observed_ptp_uv).all()),
        "amplitude_pass_rate":float(ict.amplitude_ok.mean()),
        "target_in_bounds_rate":float(ict.target_ptp_uv.between(SEIZ_TARGET_MIN_UV,SEIZ_TARGET_MAX_UV,inclusive="both").mean()),
        "boundary_mass":float(((ict.target_ptp_uv==SEIZ_TARGET_MIN_UV)|(ict.target_ptp_uv==SEIZ_TARGET_MAX_UV)).mean()),
        "median_target_error":float(ict.target_rel_error.median()),
        "generalized_morphology_rate":float((gen.generalized_frequency_ok & gen.symmetry_ok).mean()),
        "focal_morphology_rate":float((foc.focal_evolution_ok & foc.laterality_ok).mean()),
        "digest":digest.hexdigest() if include_digest else None,
    }
    report["passed"]=bool(report["finite"] and report["amplitude_pass_rate"]>=0.95 and
                          report["boundary_mass"]<0.005 and report["median_target_error"]<=0.05 and
                          report["generalized_morphology_rate"]>=0.90 and report["focal_morphology_rate"]>=0.90)
    return report,df


pilot_reports=[]; pilot_meta=[]
for seed in PILOT_SEEDS:
    report,meta=pilot_run(seed,include_digest=True)
    pilot_reports.append(report); pilot_meta.append(meta.assign(pilot_seed=seed))
    print(json.dumps(report,indent=2))

repeat_report,_=pilot_run(PILOT_SEEDS[0],include_digest=True)
reproducible = repeat_report["digest"] == pilot_reports[0]["digest"]
pilots_passed=all(r["passed"] for r in pilot_reports) and reproducible
print("Reproducibilidad seed 42:", "SUPPORTED" if reproducible else "NOT SUPPORTED")
print("Pilotos:", "SUPPORTED" if pilots_passed else "NOT SUPPORTED")
if not pilots_passed:
    raise RuntimeError("Los pilotos no cumplen los criterios predefinidos; no se genera ni sustituye el dataset final.")


{
  "seed": 42,
  "n": 100,
  "finite": true,
  "amplitude_pass_rate": 1.0,
  "target_in_bounds_rate": 1.0,
  "boundary_mass": 0.0,
  "median_target_error": 2.7748022136309557e-06,
  "generalized_morphology_rate": 1.0,
  "focal_morphology_rate": 1.0,
  "digest": "f21555cd2164406fa7b7db519036bf1f1127945efb1e1c9e7ccf960825469d66",
  "passed": true
}
{
  "seed": 142,
  "n": 100,
  "finite": true,
  "amplitude_pass_rate": 1.0,
  "target_in_bounds_rate": 1.0,
  "boundary_mass": 0.0,
  "median_target_error": 2.1683493075835497e-06,
  "generalized_morphology_rate": 1.0,
  "focal_morphology_rate": 1.0,
  "digest": "2ff9bb2c827815833802f4d1ab34ff2608c3172e413437f728443f3e7cde661d",
  "passed": true
}
{
  "seed": 242,
  "n": 100,
  "finite": true,
  "amplitude_pass_rate": 1.0,
  "target_in_bounds_rate": 1.0,
  "boundary_mass": 0.0,
  "median_target_error": 3.0241771051443925e-06,
  "generalized_morphology_rate": 1.0,
  "focal_morphology_rate": 1.0,
  "digest": "e2b1a3600bd7af2a7542bf5bb85af0fe6e

In [6]:
# BLOQUE 5 — comparación bloqueada baseline vs candidato en cinco sujetos
def dev_scores(F,y):
    rows=[]
    for subject in sorted(DEV_PROFILE["subjects"]):
        for domain,idxs in CAL_DOMAINS.items():
            gaps=[]
            classes=(0,) if domain=="background" else (1,)
            for cls in classes:
                for j in idxs:
                    name=CAL_FEATURE_NAMES[j]; q=DEV_FEATURE[(DEV_FEATURE.subject_id==subject)&(DEV_FEATURE["class"]==cls)&(DEV_FEATURE.feature==name)]
                    if q.empty: continue
                    med=float(q.iloc[0]["median"]); iqr=float(q.iloc[0]["iqr"]); floor=max(abs(med)*0.05,1e-6)
                    gaps.append(abs(float(np.median(F[y==cls,j]))-med)/(iqr+floor))
            rows.append({"subject":subject,"domain":domain,"score":float(np.median(gaps))})
        gaps=[]
        for cls in (0,1):
            for j,name in enumerate(CAL_FEATURE_NAMES):
                q=DEV_FEATURE[(DEV_FEATURE.subject_id==subject)&(DEV_FEATURE["class"]==cls)&(DEV_FEATURE.feature==name)]
                if q.empty: continue
                target=float(q.iloc[0]["iqr"]); floor=max(abs(float(q.iloc[0]["median"]))*0.05,1e-6)
                gaps.append(abs(float(stats.iqr(F[y==cls,j]))-target)/(target+floor))
        rows.append({"subject":subject,"domain":"variability","score":float(np.median(gaps))})
    return pd.DataFrame(rows)

if CALIBRATION_MODE=="frozen":
    CALIBRATION_AUDIT=FROZEN_CALIBRATION_AUDIT
    SELECT_CALIBRATED_CANDIDATE=bool(CALIBRATION_AUDIT["selected_candidate"])
    print(json.dumps(CALIBRATION_AUDIT,indent=2,ensure_ascii=False))
else:
    BASELINE_PILOT_F,BASELINE_PILOT_Y=calibration_sample(BASELINE_PARAMS)
    CANDIDATE_F,CANDIDATE_Y=calibration_sample(CALIBRATED_PARAMS)
    baseline_scores=dev_scores(BASELINE_PILOT_F,BASELINE_PILOT_Y); candidate_scores=dev_scores(CANDIDATE_F,CANDIDATE_Y)
    comparison=baseline_scores.merge(candidate_scores,on=["subject","domain"],suffixes=("_baseline","_candidate")); comparison["improved"]=comparison.score_candidate<comparison.score_baseline
    domain_summary=comparison.groupby("domain")[["score_baseline","score_candidate"]].median().reset_index(); no_domain_worse=bool((domain_summary.score_candidate<=domain_summary.score_baseline).all())
    subject_summary=comparison.groupby("subject")[["score_baseline","score_candidate"]].median(); improved_subjects=int((subject_summary.score_candidate<subject_summary.score_baseline).sum())
    ratios=[]
    for cls in (0,1):
        for j in range(len(CAL_FEATURE_NAMES)): ratios.append(stats.iqr(CANDIDATE_F[CANDIDATE_Y==cls,j])/(stats.iqr(BASELINE_PILOT_F[BASELINE_PILOT_Y==cls,j])+1e-12))
    diversity_ratio=float(np.median(ratios)); internal_pilots_ok=bool(pilots_passed)
    SELECT_CALIBRATED_CANDIDATE=bool(internal_pilots_ok and improved_subjects>=4 and no_domain_worse and diversity_ratio>=1.0-1e-9)
    CALIBRATION_AUDIT={"status":"SUPPORTED" if SELECT_CALIBRATED_CANDIDATE else "NOT SUPPORTED","selected_candidate":SELECT_CALIBRATED_CANDIDATE,
                       "improved_subjects":improved_subjects,"required_improved_subjects":4,"no_domain_worse":no_domain_worse,
                       "diversity_ratio_candidate_baseline":diversity_ratio,"pilots_passed":internal_pilots_ok,
                       "domain_scores":domain_summary.to_dict("records"),"subject_scores":subject_summary.reset_index().to_dict("records"),
                       "baseline_parameters":BASELINE_PARAMS,"candidate_parameters":CALIBRATED_PARAMS,
                       "development_profile_sha256":cal_sha256(DEV_PROFILE_PATH),"calibration_iteration":7,"previous_rejected_audit_files":["generator_calibration_audit_iteration1.json","generator_calibration_audit_iteration2.json","generator_calibration_audit_iteration3.json","generator_calibration_audit_iteration4.json","generator_calibration_audit_iteration5.json","generator_calibration_audit_iteration6.json"]}
    comparison.to_csv(DEV_PROFILE_DIR/"generator_calibration_comparison.csv",index=False)
    (DEV_PROFILE_DIR/"generator_calibration_audit.json").write_text(json.dumps(CALIBRATION_AUDIT,indent=2,ensure_ascii=False),encoding="utf-8")
    print(json.dumps(CALIBRATION_AUDIT,indent=2,ensure_ascii=False))

ACTIVE_PARAMS=CALIBRATED_PARAMS if SELECT_CALIBRATED_CANDIDATE else BASELINE_PARAMS
for name,value in ACTIVE_PARAMS.items():
    globals()[name]=tuple(value) if name in {"COMMON_FIELD_MIX","ALPHA_AMP_V","TOPO_JITTER","FOCAL_F0_RANGE","FOCAL_F1_RANGE"} else value
CONFIG["selected_profile"]="candidate" if SELECT_CALIBRATED_CANDIDATE else "baseline"
CONFIG["selected_parameters"]=ACTIVE_PARAMS


{
  "status": "NOT SUPPORTED",
  "selected_candidate": false,
  "improved_subjects": 4,
  "required_improved_subjects": 4,
  "no_domain_worse": false,
  "diversity_ratio_candidate_baseline": 1.0000722180990627,
  "pilots_passed": true,
  "domain_scores": [
    {
      "domain": "background",
      "score_baseline": 0.724129978517786,
      "score_candidate": 0.7241140989987178
    },
    {
      "domain": "spatial",
      "score_baseline": 0.7200499057155956,
      "score_candidate": 0.9155357932385582
    },
    {
      "domain": "temporal",
      "score_baseline": 1.1515498815390548,
      "score_candidate": 0.9523809523809523
    },
    {
      "domain": "variability",
      "score_baseline": 0.7246980455021901,
      "score_candidate": 0.6662917841027615
    }
  ],
  "subject_scores": [
    {
      "subject": "chb01",
      "score_baseline": 0.6939281453565087,
      "score_candidate": 0.6713080651538048
    },
    {
      "subject": "chb02",
      "score_baseline": 0.8168944772965

In [7]:
# BLOQUE 4 — generación candidata en memmap y controles integrales
def sha256_file(path, block=16*1024*1024):
    h=hashlib.sha256()
    with open(path,"rb") as f:
        while True:
            b=f.read(block)
            if not b: break
            h.update(b)
    return h.hexdigest()


def generate_candidate(n=N_PATIENTS, seed=FINAL_SEED, out_dir=CANDIDATE_DIR):
    if out_dir.exists(): shutil.rmtree(out_dir)
    out_dir.mkdir(parents=True)
    cohort=build_cohort(n,seed)
    synth=EEGSynthesizer(); patient_rows=[]; window_rows=[]
    for split_name in ("train","val","test"):
        sub=cohort[cohort.split==split_name].sort_values("patient_id")
        total=len(sub)*WINDOWS_PER_PATIENT
        X=open_memmap(out_dir/f"X_{split_name}.npy",mode="w+",dtype=np.float32,shape=(total,WIN_PTS,N_CHANNELS))
        y=np.zeros(total,dtype=np.int8); ys=np.zeros(total,dtype=np.float32)
        cursor=0
        for count,row in enumerate(sub.itertuples(index=False),1):
            data,lseq,artifact,meta=synth.generate_patient(row.scenario,seed,row.patient_id)
            meta["split"]=split_name; patient_rows.append(meta)
            for s in range(0,N_SAMPLES-WIN_PTS+1,STEP_PTS):
                frac=float(lseq[s:s+WIN_PTS].mean()); label=int(frac>=BIN_THR)
                X[cursor]=data[s:s+WIN_PTS]; y[cursor]=label; ys[cursor]=frac
                window_rows.append({"split":split_name,"window_index":cursor,"patient_id":row.patient_id,
                                    "event_id":meta["event_id"],"window_start_s":s/FS,
                                    "ictal_fraction":frac,"label":label,"scenario":row.scenario,
                                    "artifact_fraction":float(artifact[s:s+WIN_PTS].mean()),
                                    "degraded_channels":meta["degraded_channels"]})
                cursor+=1
            if count%100==0 or count==len(sub):
                print(f"{split_name}: {count}/{len(sub)} pacientes")
        X.flush(); del X
        np.save(out_dir/f"y_{split_name}.npy",y)
        np.save(out_dir/f"y_{split_name}_soft.npy",ys)
        pd.DataFrame([r for r in window_rows if r["split"]==split_name]).to_csv(out_dir/f"window_metadata_{split_name}.csv",index=False)
        del y,ys; gc.collect()
    patient_df=pd.DataFrame(patient_rows).sort_values("patient_id")
    patient_df.to_csv(out_dir/"cohort_metadata.csv",index=False)
    return patient_df,pd.DataFrame(window_rows)


def audit_candidate(out_dir, patient_df, window_df):
    issues=[]
    ids={s:set(patient_df.loc[patient_df.split==s,"patient_id"]) for s in ("train","val","test")}
    if ids["train"]&ids["val"] or ids["train"]&ids["test"] or ids["val"]&ids["test"]:
        issues.append("patient_leakage")
    ict=patient_df[patient_df.scenario!="none"]
    gen=patient_df[patient_df.scenario=="generalized_absence"]
    foc=patient_df[patient_df.scenario=="focal_temporal"]
    metrics={
        "n_patients":int(len(patient_df)),"n_windows":int(len(window_df)),
        "amplitude_pass_rate":float(ict.amplitude_ok.mean()),
        "median_target_error":float(ict.target_rel_error.median()),
        "boundary_mass":float(((ict.target_ptp_uv==SEIZ_TARGET_MIN_UV)|(ict.target_ptp_uv==SEIZ_TARGET_MAX_UV)).mean()),
        "generalized_morphology_rate":float((gen.generalized_frequency_ok & gen.symmetry_ok).mean()),
        "focal_morphology_rate":float((foc.focal_evolution_ok & foc.laterality_ok).mean()),
        "patient_split_disjoint":not bool(issues),
    }
    for split_name in ("train","val","test"):
        X=np.load(out_dir/f"X_{split_name}.npy",mmap_mode="r")
        y=np.load(out_dir/f"y_{split_name}.npy")
        expected=len(ids[split_name])*WINDOWS_PER_PATIENT
        if X.shape!=(expected,WIN_PTS,N_CHANNELS) or len(y)!=expected: issues.append(f"shape_{split_name}")
        for a in range(0,len(X),2048):
            if not np.isfinite(np.asarray(X[a:a+2048])).all():
                issues.append(f"nonfinite_{split_name}"); break
        metrics[f"{split_name}_shape"]=list(X.shape)
        metrics[f"{split_name}_ictal_fraction"]=float(y.mean())
        del X,y
    passed=(not issues and metrics["amplitude_pass_rate"]>=0.95 and metrics["median_target_error"]<=0.05 and
            metrics["boundary_mass"]<0.005 and metrics["generalized_morphology_rate"]>=0.90 and
            metrics["focal_morphology_rate"]>=0.90)
    return {"status":"SUPPORTED" if passed else "NOT SUPPORTED","passed":bool(passed),"issues":issues,"metrics":metrics}


def verify_existing_dataset(final_dir):
    manifest_path=final_dir/"generation_manifest.json"
    required=[f"X_{s}.npy" for s in ("train","val","test")]+[f"y_{s}.npy" for s in ("train","val","test")]+[f"window_metadata_{s}.csv" for s in ("train","val","test")]+["cohort_metadata.csv"]
    if not manifest_path.exists() or not all((final_dir/n).exists() for n in required): return False,"faltan artefactos"
    manifest=json.loads(manifest_path.read_text(encoding="utf-8")); files=manifest.get("files",{})
    bad=[n for n in required if n not in files or sha256_file(final_dir/n)!=files[n].get("sha256")]
    if bad: return False,"hash inválido: "+", ".join(bad[:4])
    selected=manifest.get("external_freeze",{}).get("selected","candidate")
    expected="candidate" if SELECT_CALIBRATED_CANDIDATE else "baseline"
    if selected!=expected: return False,f"perfil existente {selected}, esperado {expected}"
    return True,"manifiesto y hashes válidos"

REUSED_EXISTING=False
if RUN_FULL:
    REUSED_EXISTING,reason=verify_existing_dataset(FINAL_DIR)
    if REUSED_EXISTING:
        existing_manifest=json.loads((FINAL_DIR/"generation_manifest.json").read_text(encoding="utf-8"))
        audit=existing_manifest.get("candidate_audit",{"status":"SUPPORTED","passed":True,"reason":"dataset publicado verificado"})
        audit["passed"]=True
        print("Dataset SYN existente: reutilizado;",reason)
    else:
        print("Dataset SYN: reconstrucción completa activada;",reason)
        candidate_patients,candidate_windows=generate_candidate()
        audit=audit_candidate(CANDIDATE_DIR,candidate_patients,candidate_windows)
        print(json.dumps(audit,indent=2))
        if not audit["passed"]: raise RuntimeError("El dataset reconstruido no cumple el contrato; no se promueve.")
else:
    audit={"status":"NOT EVALUABLE","passed":False,"reason":"GENERATION_MODE=pilot"}
    print("Generación completa omitida: los pilotos constituyen la salida de este modo")


Dataset SYN existente: reutilizado; manifiesto y hashes válidos


In [8]:
# BLOQUE 5 — figuras representativas, manifiesto y promoción atómica
def compact_medoid_indices(X,y,seed=42,n_pool=500):
    rng=np.random.default_rng(seed); out={}
    for cls in (0,1):
        ids=np.flatnonzero(y==cls); pick=rng.choice(ids,min(n_pool,len(ids)),replace=False)
        feats=[]
        for i in pick:
            w=np.asarray(X[i],dtype=np.float64)
            ptp=np.median(np.ptp(w,axis=0)); rms=np.median(np.sqrt(np.mean(w*w,axis=0)))
            f,p=signal.welch(w.mean(axis=1),fs=FS,nperseg=WIN_PTS)
            bp=lambda a,b:np.trapezoid(p[(f>=a)&(f<b)],f[(f>=a)&(f<b)])
            feats.append([np.log1p(ptp),np.log1p(rms),np.log1p(bp(1,4)),np.log1p(bp(4,8)),np.log1p(bp(13,30))])
        F=np.asarray(feats); z=(F-np.median(F,axis=0))/(stats.median_abs_deviation(F,axis=0,scale="normal")+1e-9)
        center=np.median(z,axis=0); out[cls]=int(pick[np.argmin(np.sum((z-center)**2,axis=1))])
    return out


def build_figures(out_dir):
    X=np.load(out_dir/"X_train.npy",mmap_mode="r"); y=np.load(out_dir/"y_train.npy")
    wm=pd.read_csv(out_dir/"window_metadata_train.csv")
    rng=np.random.default_rng(42)
    def medoid(indices, channel):
        indices=np.asarray(indices,dtype=int); pick=rng.choice(indices,min(400,len(indices)),replace=False); feats=[]
        for i in pick:
            w=np.asarray(X[i,:,channel],dtype=np.float64); f,p=signal.welch(w,fs=FS,nperseg=WIN_PTS)
            def bp(a,b):
                m=(f>=a)&(f<b); return np.trapezoid(p[m],f[m]) if m.any() else 0.0
            feats.append([np.ptp(w),np.std(w),np.log1p(bp(1,4)),np.log1p(bp(4,8)),np.log1p(bp(8,13))])
        F=np.asarray(feats); z=(F-np.median(F,axis=0))/(stats.median_abs_deviation(F,axis=0,scale="normal")+1e-9)
        return int(pick[np.argmin(np.sum((z-np.median(z,axis=0))**2,axis=1))])
    groups={"Interictal":(wm.index[wm.label==0].to_numpy(),CH_NAMES.index("Cz")),
            "Generalized - frontocentral":(wm.index[(wm.label==1)&(wm.scenario=="generalized_absence")].to_numpy(),CH_NAMES.index("Fz")),
            "Focal - temporal":(wm.index[(wm.label==1)&(wm.scenario=="focal_temporal")].to_numpy(),CH_NAMES.index("T3"))}
    ids={name:medoid(ix,ch) for name,(ix,ch) in groups.items() if len(ix)>0}; t=np.arange(WIN_PTS)/FS
    fig,ax=plt.subplots(3,1,figsize=(10,7),sharex=True)
    for a,(name,i) in zip(ax,ids.items()):
        ch=groups[name][1]; a.plot(t,np.asarray(X[i,:,ch]),lw=0.9); a.set_ylabel("uV"); a.set_title(f"{name} - {CH_NAMES[ch]}"); a.grid(alpha=.25)
    ax[-1].set_xlabel("Time (s)"); fig.tight_layout(); fig.savefig(out_dir/"fig1_eeg_waveforms.png",dpi=180); fig.savefig(out_dir/"fig1_eeg_waveforms.pdf",metadata={"Creator":"EEGSynthesizer","CreationDate":None,"ModDate":None}); plt.close(fig)
    fig,ax=plt.subplots(figsize=(8,4))
    for name,i in ids.items():
        ch=groups[name][1]; f,p=signal.welch(np.asarray(X[i,:,ch],dtype=np.float64),fs=FS,nperseg=WIN_PTS); ax.semilogy(f,p,label=name)
    ax.set_xlim(1,30); ax.set_xlabel("Hz"); ax.set_ylabel("PSD (uV^2/Hz)"); ax.grid(alpha=.25); ax.legend(fontsize=8); fig.tight_layout(); fig.savefig(out_dir/"fig2_psd_comparison.png",dpi=180); fig.savefig(out_dir/"fig2_psd_comparison.pdf",metadata={"Creator":"EEGSynthesizer","CreationDate":None,"ModDate":None}); plt.close(fig)
    ids0=wm.index[wm.label==0].to_numpy(); ids1=wm.index[wm.label==1].to_numpy(); ids0=rng.choice(ids0,min(500,len(ids0)),replace=False); ids1=rng.choice(ids1,min(500,len(ids1)),replace=False)
    med0=np.median([np.ptp(np.asarray(X[i]),axis=0) for i in ids0],axis=0); med1=np.median([np.ptp(np.asarray(X[i]),axis=0) for i in ids1],axis=0); ratio=med1/(med0+1e-9)
    fig,ax=plt.subplots(figsize=(11,4)); ax.bar(CH_NAMES,ratio,color=plt.cm.viridis((ratio-ratio.min())/(np.ptp(ratio)+1e-12))); ax.axhline(1,color="black",lw=.8); ax.set_ylabel("PTP ictal / interictal"); ax.set_title(f"Topography - full range {ratio.min():.2f}-{ratio.max():.2f}"); ax.tick_params(axis="x",rotation=45); fig.tight_layout(); fig.savefig(out_dir/"fig3_topography.png",dpi=180); fig.savefig(out_dir/"fig3_topography.pdf",metadata={"Creator":"EEGSynthesizer","CreationDate":None,"ModDate":None}); plt.close(fig)
    pd.DataFrame({"channel":CH_NAMES,"ptp_interictal_uv":med0,"ptp_ictal_uv":med1,"ratio":ratio}).to_csv(out_dir/"tabla_amplitudes.csv",index=False); del X,y,wm


def promote_candidate(candidate,final_dir):
    """Promoción segura de directorios en Windows: cerrar memmaps y renombrar."""
    gc.collect()
    previous=WORK_ROOT/"dataset_eeg_previous"
    if previous.exists(): shutil.rmtree(previous)
    had_final = final_dir.exists()
    if had_final: final_dir.rename(previous)
    try:
        candidate.rename(final_dir)
    except Exception:
        if final_dir.exists(): shutil.rmtree(final_dir)
        if previous.exists(): previous.rename(final_dir)
        raise
    if previous.exists(): shutil.rmtree(previous)


if RUN_FULL and not REUSED_EXISTING and audit["passed"]:
    build_figures(CANDIDATE_DIR)
    pd.DataFrame(pilot_reports).to_csv(CANDIDATE_DIR/"pilot_validation.csv",index=False)
    manifest={"config":CONFIG,"pilot_reports":pilot_reports,"candidate_audit":audit,"development_calibration":CALIBRATION_AUDIT,
              "external_freeze":{"frozen":True,"frozen_utc":time.strftime("%Y-%m-%dT%H:%M:%SZ",time.gmtime()),"selected":CONFIG["selected_profile"]},
              "software":{"python":sys.version,"platform":platform.platform(),"numpy":np.__version__,"scipy":scipy.__version__,"pandas":pd.__version__},
              "created_utc":time.strftime("%Y-%m-%dT%H:%M:%SZ",time.gmtime()),"files":{}}
    for p in sorted(CANDIDATE_DIR.iterdir()):
        if p.is_file() and p.name!="generation_manifest.json": manifest["files"][p.name]={"bytes":p.stat().st_size,"sha256":sha256_file(p)}
    part=CANDIDATE_DIR/"generation_manifest.json.part"; part.write_text(json.dumps(manifest,indent=2,ensure_ascii=False)+"\n",encoding="utf-8"); os.replace(part,CANDIDATE_DIR/"generation_manifest.json")
    promote_candidate(CANDIDATE_DIR,FINAL_DIR)
    print("Dataset promovido atómicamente:",FINAL_DIR)
    print("Resultado generador: SUPPORTED —",CONFIG["selected_profile"],"reconstruido")
elif RUN_FULL and REUSED_EXISTING:
    print("Resultado generador: SUPPORTED — dataset publicado reutilizado sin modificación")
else:
    print("Resultado generador: NOT EVALUABLE en modo pilot")

if RUN_FULL:
    state={"schema_version":"1.0-reproducibility","stages":{}}
    if STATE_PATH.exists():
        try: state=json.loads(STATE_PATH.read_text(encoding="utf-8"))
        except Exception: pass
    state.setdefault("stages",{})["S3_synthetic_frozen"]={
      "status":"SUPPORTED","completed_utc":time.strftime("%Y-%m-%dT%H:%M:%SZ",time.gmtime()),
      "details":{"selected_profile":CONFIG["selected_profile"],"reused":REUSED_EXISTING},
      "artifacts":{"generation_manifest":{"bytes":(FINAL_DIR/"generation_manifest.json").stat().st_size,"sha256":sha256_file(FINAL_DIR/"generation_manifest.json")}}}
    STATE_PATH.parent.mkdir(parents=True,exist_ok=True); part=Path(str(STATE_PATH)+".part")
    part.write_text(json.dumps(state,indent=2,ensure_ascii=False)+"\n",encoding="utf-8"); os.replace(part,STATE_PATH)
    print("Reanudación S3 registrada en",STATE_PATH)


Resultado generador: SUPPORTED — dataset publicado reutilizado sin modificación
Reanudación S3 registrada en C:\Users\ediso\Documents\EDISON MENESES\DOCTORADO\TESIS\EEGSynthesizer\dataset_doctorado_final\reproducibility_state.json


## Auditoría post-hoc de simetría generalizada

Este bloque **no modifica ni recalibra el generador**. Reconstruye cada evento
`generalized_absence` desde las ventanas finales almacenadas y separa dos preguntas:

1. `clean_nondegraded`: simetría de los pares frontales que no fueron degradados;
2. `final_all_channels`: simetría observable incluyendo pares afectados por degradación de adquisición.

El criterio operacional es la mediana de la asimetría relativa de `Fp1–Fp2` y
`F3–F4` menor o igual a 0.35. Los resultados son verificaciones de la señal sintética
final, no validación clínica.

In [9]:
if GENERATION_MODE!="full":
    print("Auditoría post-hoc: NOT EVALUABLE en modo pilot")
else:
    # AUDITORÍA POST-HOC DE SIMETRÍA GENERALIZADA — no recalibra el generador
    import hashlib
    import json
    import time
    from pathlib import Path

    import numpy as np
    import pandas as pd
    from scipy import signal

    AUDIT_ROOT = Path.cwd()
    AUDIT_SYN_DIR = AUDIT_ROOT / "dataset_eeg_final"
    AUDIT_OUT_DIR = AUDIT_ROOT / "dataset_doctorado_final" / "validation_q1_assets"
    AUDIT_OUT_DIR.mkdir(parents=True, exist_ok=True)
    AUDIT_DETAIL_PATH = AUDIT_OUT_DIR / "07_generalized_symmetry_by_patient.csv"
    AUDIT_SUMMARY_PATH = AUDIT_OUT_DIR / "07_generalized_symmetry_summary.json"

    AUDIT_FS = 250
    AUDIT_CHANNELS = ['Fp1','Fp2','F7','F3','Fz','F4','F8',
                      'T3','C3','Cz','C4','T4','T5','P3','Pz','P4','T6','O1','O2']
    AUDIT_INDEX = {name: idx for idx, name in enumerate(AUDIT_CHANNELS)}
    AUDIT_PAIRS = [('Fp1','Fp2'), ('F3','F4')]
    AUDIT_PRIMARY = {'Fp1','Fp2','F3','Fz','F4'}
    AUDIT_SYMMETRY_THRESHOLD = 0.35
    AUDIT_ACCEPTANCE_RATE = 0.90

    def audit_sha256(path, block=16*1024*1024):
        digest = hashlib.sha256()
        with Path(path).open("rb") as stream:
            for chunk in iter(lambda: stream.read(block), b""):
                digest.update(chunk)
        return digest.hexdigest()

    def audit_dominant_frequency(x, fs, low=2.0, high=8.0):
        freq, power = signal.welch(np.asarray(x, dtype=np.float64), fs=fs,
                                   nperseg=min(len(x), 2*fs))
        mask = (freq >= low) & (freq <= high)
        return float(freq[mask][np.argmax(power[mask])])

    manifest_path = AUDIT_SYN_DIR / "generation_manifest.json"
    manifest = json.loads(manifest_path.read_text(encoding="utf-8"))
    required = ["cohort_metadata.csv"] + [
        name for split in ("train", "val", "test")
        for name in (f"X_{split}.npy", f"window_metadata_{split}.csv")
    ]
    missing = [name for name in required if not (AUDIT_SYN_DIR/name).exists()]
    if missing:
        raise RuntimeError(f"Faltan entradas de la auditoría: {missing}")

    hash_failures = []
    for name in required:
        recorded = manifest.get("files", {}).get(name, {}).get("sha256")
        observed = audit_sha256(AUDIT_SYN_DIR/name)
        if not recorded or observed != recorded:
            hash_failures.append(name)
    if hash_failures:
        raise RuntimeError(f"SHA256 de entradas inválido: {hash_failures}")

    cohort = pd.read_csv(AUDIT_SYN_DIR/"cohort_metadata.csv")
    generalized = cohort[cohort.scenario == "generalized_absence"].copy()
    if len(generalized) != 750:
        raise RuntimeError(f"Se esperaban 750 eventos generalizados y hay {len(generalized)}")

    sos = signal.butter(4, [1.0, 30.0], btype="bandpass", fs=AUDIT_FS, output="sos")
    rows = []
    for split in ("train", "val", "test"):
        X = np.load(AUDIT_SYN_DIR/f"X_{split}.npy", mmap_mode="r")
        wm = pd.read_csv(AUDIT_SYN_DIR/f"window_metadata_{split}.csv")
        groups = {int(pid): group.sort_values("window_start_s")
                  for pid, group in wm.groupby("patient_id", sort=False)}
        split_events = generalized[generalized.split == split]
        for record in split_events.itertuples(index=False):
            group = groups.get(int(record.patient_id))
            if group is None or len(group) != 60:
                raise RuntimeError(f"Ventanas incompletas para paciente {record.patient_id}")
            starts = group.window_start_s.to_numpy(float)
            if not np.allclose(starts, np.arange(60)*2.0):
                raise RuntimeError(f"Secuencia temporal inválida para paciente {record.patient_id}")
            full = np.concatenate([np.asarray(X[int(idx)])
                                   for idx in group.window_index], axis=0).T
            i0 = int(round(float(record.start_s)*AUDIT_FS))
            i1 = min(full.shape[1], i0 + int(round(float(record.duration_s)*AUDIT_FS)))
            if i1-i0 < 3*12:
                raise RuntimeError(f"Evento demasiado corto para paciente {record.patient_id}")
            event = signal.sosfiltfilt(sos, full[:, i0:i1], axis=1)
            ptp = np.ptp(event, axis=1)
            degraded = (set(str(record.degraded_channels).split("|"))
                        if pd.notna(record.degraded_channels) else set())
            pair_values = {}
            clean_values = []
            for left, right in AUDIT_PAIRS:
                value = float(abs(ptp[AUDIT_INDEX[left]]-ptp[AUDIT_INDEX[right]]) /
                              (max(ptp[AUDIT_INDEX[left]], ptp[AUDIT_INDEX[right]])+1e-12))
                pair_values[f"{left}_{right}"] = value
                if left not in degraded and right not in degraded:
                    clean_values.append(value)
            all_asymmetry = float(np.median(list(pair_values.values())))
            clean_asymmetry = (float(np.median(clean_values)) if clean_values else float("nan"))
            clean_ok = bool(not clean_values or clean_asymmetry <= AUDIT_SYMMETRY_THRESHOLD)
            final_ok = bool(all_asymmetry <= AUDIT_SYMMETRY_THRESHOLD)
            frequency = audit_dominant_frequency(event[AUDIT_INDEX["Fz"]], AUDIT_FS)
            peak_channel = AUDIT_CHANNELS[int(np.argmax(ptp))]
            rows.append({
                "patient_id": int(record.patient_id), "split": split,
                "degraded_channels": "|".join(sorted(degraded)),
                "frontal_pair_degraded": bool(any(ch in degraded for pair in AUDIT_PAIRS for ch in pair)),
                "asymmetry_Fp1_Fp2": pair_values["Fp1_Fp2"],
                "asymmetry_F3_F4": pair_values["F3_F4"],
                "clean_nondegraded_asymmetry": clean_asymmetry,
                "final_all_channels_asymmetry": all_asymmetry,
                "clean_nondegraded_symmetry_ok": clean_ok,
                "final_all_channels_symmetry_ok": final_ok,
                "dominant_frequency_fz_hz": frequency,
                "frequency_2_5_4_ok": bool(2.5 <= frequency <= 4.0),
                "peak_channel": peak_channel,
                "frontcentral_peak_ok": bool(peak_channel in AUDIT_PRIMARY),
            })

    detail = pd.DataFrame(rows).sort_values("patient_id").reset_index(drop=True)
    if len(detail) != len(generalized) or detail.patient_id.duplicated().any():
        raise RuntimeError("Cobertura de pacientes inválida en auditoría de simetría")
    if not np.isfinite(detail[["asymmetry_Fp1_Fp2", "asymmetry_F3_F4",
                               "final_all_channels_asymmetry",
                               "dominant_frequency_fz_hz"]].to_numpy()).all():
        raise RuntimeError("NaN/Inf en resultados de simetría")

    detail.to_csv(AUDIT_DETAIL_PATH, index=False)
    clean_rate = float(detail.clean_nondegraded_symmetry_ok.mean())
    final_rate = float(detail.final_all_channels_symmetry_ok.mean())
    summary = {
        "schema_version": "1.0-symmetry-audit",
        "generated_utc": time.strftime("%Y-%m-%dT%H:%M:%SZ", time.gmtime()),
        "source_generation_manifest_sha256": audit_sha256(manifest_path),
        "validator_notebook_sha256": audit_sha256(AUDIT_ROOT/"1_0_EEGSynthesizer_DATASET.ipynb"),
        "detail_csv": AUDIT_DETAIL_PATH.name,
        "detail_csv_sha256": audit_sha256(AUDIT_DETAIL_PATH),
        "source_hashes_verified": True,
        "generator_parameters_changed": False,
        "generator_recalibrated": False,
        "external_cohort_used": False,
        "n_generalized_events": int(len(detail)),
        "symmetry_relative_difference_threshold": AUDIT_SYMMETRY_THRESHOLD,
        "acceptance_rate": AUDIT_ACCEPTANCE_RATE,
        "clean_nondegraded": {
            "status": "SUPPORTED" if clean_rate >= AUDIT_ACCEPTANCE_RATE else "NOT SUPPORTED",
            "pass_count": int(detail.clean_nondegraded_symmetry_ok.sum()),
            "pass_rate": clean_rate,
            "interpretation": "simetría morfológica en pares frontales no degradados",
        },
        "final_all_channels": {
            "status": "SUPPORTED" if final_rate >= AUDIT_ACCEPTANCE_RATE else "NOT SUPPORTED",
            "pass_count": int(detail.final_all_channels_symmetry_ok.sum()),
            "pass_rate": final_rate,
            "interpretation": "simetría observable después de degradación de adquisición",
        },
        "events_with_frontal_pair_degradation": int(detail.frontal_pair_degraded.sum()),
        "frequency_2_5_4": {
            "pass_count": int(detail.frequency_2_5_4_ok.sum()),
            "pass_rate": float(detail.frequency_2_5_4_ok.mean()),
        },
        "frontcentral_peak": {
            "pass_count": int(detail.frontcentral_peak_ok.sum()),
            "pass_rate": float(detail.frontcentral_peak_ok.mean()),
        },
        "overall_interpretation": (
            "La morfología bilateral está respaldada en pares no degradados; "
            "la simetría final debe reportarse por separado porque la degradación puede romperla."
        ),
    }
    AUDIT_SUMMARY_PATH.write_text(json.dumps(summary, indent=2, ensure_ascii=False)+"\n",
                                  encoding="utf-8")
    print(json.dumps(summary, indent=2, ensure_ascii=False))


{
  "schema_version": "1.0-symmetry-audit",
  "generated_utc": "2026-08-04T11:15:34Z",
  "source_generation_manifest_sha256": "08091538c5cb593ea711139db512145ef505b0bbc2dcface273807434a7b2db0",
  "validator_notebook_sha256": "b5c29932760471a2ff442f86ae362d40aa7fb049fbe4d1418d3abd5a90649a2d",
  "detail_csv": "07_generalized_symmetry_by_patient.csv",
  "detail_csv_sha256": "057555292a2cd35a0be15137145b6fda0a46bf27632d6609fc96f4bc65f0498d",
  "source_hashes_verified": true,
  "generator_parameters_changed": false,
  "generator_recalibrated": false,
  "external_cohort_used": false,
  "n_generalized_events": 750,
  "symmetry_relative_difference_threshold": 0.35,
  "acceptance_rate": 0.9,
  "clean_nondegraded": {
    "status": "SUPPORTED",
    "pass_count": 750,
    "pass_rate": 1.0,
    "interpretation": "simetría morfológica en pares frontales no degradados"
  },
  "final_all_channels": {
    "status": "NOT SUPPORTED",
    "pass_count": 603,
    "pass_rate": 0.804,
    "interpretation": "s